# Circadian Metrics Notebook

This notebook downloads or clones the GitHub repo, points the analysis at the repo data folder, and runs `Circadian_Metrics_IV_IS_RA.py` without requiring a hardcoded local path.

In [ ]:
from pathlib import Path
import runpy
import shutil
import subprocess
import tempfile
import urllib.request
import zipfile
from unittest.mock import patch

In [ ]:
REPO_URL = "https://github.com/Oscar-Pineda524/CAAP193.git"
BRANCH_CANDIDATES = ["main", "master"]
NOTEBOOK_ROOT = Path.cwd()
REPO_DIR = NOTEBOOK_ROOT / "CAAP193_repo"
DATA_SUBDIR = "60s"
FORCE_REFRESH = False

In [ ]:
def ensure_repo(repo_url: str, repo_dir: Path, branches: list[str], force_refresh: bool = False) -> Path:
    repo_dir = repo_dir.resolve()

    if force_refresh and repo_dir.exists():
        shutil.rmtree(repo_dir)

    if repo_dir.exists():
        print(f"Using existing repo at {repo_dir}")
        return repo_dir

    repo_dir.parent.mkdir(parents=True, exist_ok=True)

    try:
        subprocess.run(
            ["git", "clone", repo_url, str(repo_dir)],
            check=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
        )
        print(f"Cloned repo to {repo_dir}")
        return repo_dir
    except Exception as clone_error:
        print(f"git clone failed: {clone_error}")
        print("Falling back to downloading a zip archive from GitHub...")

    for branch in branches:
        zip_url = repo_url.removesuffix(".git") + f"/archive/refs/heads/{branch}.zip"
        try:
            with tempfile.TemporaryDirectory() as tmpdir:
                zip_path = Path(tmpdir) / f"{branch}.zip"
                urllib.request.urlretrieve(zip_url, zip_path)

                with zipfile.ZipFile(zip_path) as zf:
                    zf.extractall(tmpdir)

                extracted_root = next(Path(tmpdir).glob("*"))
                shutil.move(str(extracted_root), str(repo_dir))
                print(f"Downloaded repo archive ({branch}) to {repo_dir}")
                return repo_dir
        except Exception as zip_error:
            print(f"Zip download failed for branch '{branch}': {zip_error}")

    raise RuntimeError("Could not fetch the repo by git clone or zip download.")

In [ ]:
repo_dir = ensure_repo(REPO_URL, REPO_DIR, BRANCH_CANDIDATES, force_refresh=FORCE_REFRESH)
script_path = repo_dir / "Circadian_Metrics_IV_IS_RA.py"
data_root = repo_dir / DATA_SUBDIR
surgery_dates_path = repo_dir / "surgery_dates.csv"

if not script_path.exists():
    raise FileNotFoundError(f"Could not find script: {script_path}")

if not surgery_dates_path.exists():
    raise FileNotFoundError(f"Could not find surgery_dates.csv: {surgery_dates_path}")

if not data_root.exists():
    raise FileNotFoundError(f"Could not find data folder: {data_root}")

print(f"Repo directory: {repo_dir}")
print(f"Script path: {script_path}")
print(f"Data folder: {data_root}")

In [ ]:
import pandas as pd

pd.read_csv(surgery_dates_path).head()

In [ ]:
with patch("builtins.input", return_value=str(data_root)):
    runpy.run_path(str(script_path), run_name="__main__")

## Notes

- Change `DATA_SUBDIR` if the repo stores the actigraphy files somewhere besides `60s`.
- Set `FORCE_REFRESH = True` if you want the notebook to redownload the repo.
- The script writes outputs into the cloned/downloaded repo directory because it keeps the original script behavior.